# Behavioral Pattern, notebook 2: threshold calibration

Rule fixed in the protocol before any agreement was computed: the proposal's cut points read as cohort quantiles. 0.7 becomes the 70th percentile, 0.3 the 30th, 0.4 the 40th.
E and C use the 144 real student-sprints. Collaboration uses the sprints where it is available, real and simulated together (the real students alone give 8). The numbers are
written to `thresholds.json` next to `rules.py` so the agent uses fixed numbers.

In [1]:
import sys, warnings, json, importlib.util
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, confusion_matrix
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)
ROOT = Path("../../..").resolve()
spec = importlib.util.spec_from_file_location("bp_rules_nb", ROOT / "rules.py")
R = importlib.util.module_from_spec(spec); spec.loader.exec_module(R)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = pd.read_csv(Path("../../dataset/bp_inputs.csv"))
PATTERNS = list(R.PATTERNS) + ["Balanced contributor"]
def run(thresholds, frame=None):
    frame = d if frame is None else frame
    out = []
    for r in frame.itertuples():
        co = None if pd.isna(r.Co) else float(r.Co)
        bim = None if pd.isna(r.bimodal) else int(r.bimodal)
        lc = None if pd.isna(r.late_churn) else float(r.late_churn)
        out.append(R.classify({"E": float(r.E), "C": float(r.C), "Co": co}, bim, thresholds, lc))
    return out

In [2]:
real = d[d.source == "real"]
co = d[d.Co_available].Co.dropna()
cal = {"E_high": real.E.quantile(0.70), "C_high": real.C.quantile(0.70), "C_low": real.C.quantile(0.30), "E_mid": real.E.quantile(0.40), "E_low": real.E.quantile(0.30),
       "Co_high": R.PROPOSAL["Co_high"], "Co_low": R.PROPOSAL["Co_low"]}   # amendment 1 in PROTOCOL.md: the quantile reading gave Co_high = Co_low = 0.25 (ties)
cal = {k: round(float(v), 4) for k, v in cal.items()}
quantile_co = (round(float(co.quantile(0.70)), 4), round(float(co.quantile(0.30)), 4))
print("Collaboration cut points the quantile reading would have given (not used):", quantile_co)
pd.DataFrame({"proposal": {k: v for k, v in R.PROPOSAL.items() if k != "name"}, "calibrated": cal})

Collaboration cut points the quantile reading would have given (not used): (0.25, 0.25)


,proposal,calibrated
E_high,0.7,0.5431
C_high,0.7,0.3184
C_low,0.3,0.0534
E_mid,0.4,0.3091
E_low,0.3,0.1975
Co_high,0.7,0.7000
Co_low,0.3,0.3000


In [3]:
print("n for E and C:", len(real), "; n for Collaboration:", len(co), "; distinct Collaboration values:", co.round(4).nunique())
print(co.round(3).value_counts().head(6).to_dict())
print("quantile reading degenerate (tied values):", quantile_co[0] <= quantile_co[1] + 0.02)

n for E and C: 144 ; n for Collaboration: 133 ; distinct Collaboration values: 33
{0.25: 77, 0.0: 15, 0.2: 11, 0.791: 1, 0.449: 1, 0.772: 1}
quantile reading degenerate (tied values): True


In [4]:
active = real[real.late_churn.notna()]
ext = {"E_med": round(float(real.E.quantile(0.50)), 4), "DC_high": round(float(active.late_churn.quantile(0.70)), 4), "ext_rusher": True, "ext_free_rider": True}   # amendments A and B in PROTOCOL.md; the decision below (notebook 3) selects what "calibrated" keeps
print("extension cut points:", ext, "(late-churn share from", len(active), "active real student-sprints)")
decision = json.loads((RESULTS / "extension_decision.json").read_text(encoding="utf-8")) if (RESULTS / "extension_decision.json").exists() else {"A kept": False, "B kept": True}
adopted = {**cal, "E_med": ext["E_med"], "DC_high": ext["DC_high"], "ext_rusher": bool(decision.get("A kept")), "ext_free_rider": bool(decision.get("B kept"))}
print("decision used for the stored default set:", {"ext_rusher": adopted["ext_rusher"], "ext_free_rider": adopted["ext_free_rider"]})
meta = {"sets": {"calibrated_v1": cal, "calibrated_AB": {**cal, **ext}, "calibrated": adopted}, "calibrated_from": {"E_C": f"{len(real)} real student-sprints (marks in dataset/clean/final_dataset_clean.csv)", "Co": "the proposal's absolute 0.7 / 0.3 (the quantile reading was degenerate, see the amendment log)",
        "rule": "Effort and Consistency: proposal cut points read as cohort quantiles (0.7 -> P70, 0.3 -> P30, 0.4 -> P40); Collaboration: the proposal's own 0.7 / 0.3", "date": "2026-10-19", "extensions": "calibrated_v1 = original rules; calibrated_AB = both extensions of the protocol; calibrated = the extensions that passed the keep/revert check of notebook 3 (results/extension_decision.json)"}}
(ROOT / "thresholds.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")
(RESULTS / "thresholds.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")
meta

extension cut points: {'E_med': 0.422, 'DC_high': 0.2622, 'ext_rusher': True, 'ext_free_rider': True} (late-churn share from 104 active real student-sprints)
decision used for the stored default set: {'ext_rusher': False, 'ext_free_rider': True}


{'sets': {'calibrated_v1': {'E_high': 0.5431,
   'C_high': 0.3184,
   'C_low': 0.0534,
   'E_mid': 0.3091,
   'E_low': 0.1975,
   'Co_high': 0.7,
   'Co_low': 0.3},
  'calibrated_AB': {'E_high': 0.5431,
   'C_high': 0.3184,
   'C_low': 0.0534,
   'E_mid': 0.3091,
   'E_low': 0.1975,
   'Co_high': 0.7,
   'Co_low': 0.3,
   'E_med': 0.422,
   'DC_high': 0.2622,
   'ext_rusher': True,
   'ext_free_rider': True},
  'calibrated': {'E_high': 0.5431,
   'C_high': 0.3184,
   'C_low': 0.0534,
   'E_mid': 0.3091,
   'E_low': 0.1975,
   'Co_high': 0.7,
   'Co_low': 0.3,
   'E_med': 0.422,
   'DC_high': 0.2622,
   'ext_rusher': False,
   'ext_free_rider': True}},
 'calibrated_from': {'E_C': '144 real student-sprints (marks in dataset/clean/final_dataset_clean.csv)',
  'Co': "the proposal's absolute 0.7 / 0.3 (the quantile reading was degenerate, see the amendment log)",
  'rule': "Effort and Consistency: proposal cut points read as cohort quantiles (0.7 -> P70, 0.3 -> P30, 0.4 -> P40); Collaborati

## Reading the results

**Calibrated cut points** (written to `thresholds.json`): Effort high 0.543, mid 0.309, low 0.198; Consistency high 0.318, low 0.053; Collaboration 0.7 / 0.3.

**Two things the calibration showed.**
1. The quantile reading of Collaboration was degenerate: 77 of the 133 available scores equal 0.25 (the response-rate measure of the Collaboration score gives that value to anyone who only merged their own pull requests), so the 30th and 70th percentiles are both 0.25. A Reviewer rule (above 0.25) and a Free-rider rule (below 0.25) cut at the same point would be meaningless. Collaboration therefore keeps the proposal's absolute 0.7 / 0.3; this change was made before any agreement figure was computed and is in the amendment log of the protocol.
2. The Consistency low cut is 0.053 because a quarter of the real student-sprints have a Consistency of exactly 0 (no activity). A rule that needs Consistency below 0.053 together with Effort above 0.543 can hardly be met, which matters for the Last-minute rusher rule (see the next notebook).

These cut points describe the 144 real student-sprints of one course. They are fixed numbers now and will not move when other students are scored; they should be recomputed, with the same stated rule, when a new cohort is added.